In [ ]:
# YelpChi 数据集 Baseline: H2GCN (Heterogeneous Version)
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import scipy.io as sio 
from torch_geometric.data import HeteroData  
from torch_geometric.nn import HeteroConv, GCNConv
from torch.cuda.amp import autocast, GradScaler 
from sklearn.metrics import confusion_matrix, f1_score, roc_auc_score

# ==========================================
# 【模块一】：Config 参数 (保持对齐)
# ==========================================
class Config:
    hidden_dim = 32      
    epochs = 100         
    lr_init = 0.001
    weight_decay = 1e-4

class EarlyStopping:
    def __init__(self, patience=15, min_delta=0):
        self.patience, self.min_delta = patience, min_delta
        self.counter, self.best_score, self.early_stop = 0, None, False
    def __call__(self, current_f1):
        if self.best_score is None: self.best_score = current_f1
        elif current_f1 < self.best_score + self.min_delta:
            self.counter += 1
            if self.counter >= self.patience: self.early_stop = True
        else:
            self.best_score, self.counter = current_f1, 0

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ==========================================
# 【模块二】：H2GCN 核心架构
# ==========================================
class H2GCN_Layer(nn.Module):
    """ 实现 H2GCN 的异质扩展：分离自身特征与一阶邻居聚合特征 """
    def __init__(self, metadata, in_channels, out_channels):
        super().__init__()
        # 使用 HeteroConv 包装基础 GCN
        self.conv = HeteroConv({
            rel: GCNConv(in_channels, out_channels, add_self_loops=False)
            for rel in metadata[1]
        }, aggr='sum')

    def forward(self, x_dict, edge_index_dict):
        # 聚合邻居特征 (Eq. 2 in H2GCN: a_v^(k))
        h_neighbors = self.conv(x_dict, edge_index_dict)
        return h_neighbors

class H2GCN_Hetero(nn.Module):
    def __init__(self, metadata, in_side_dict, hidden_size, out_size):
        super().__init__()
        self.node_types = metadata[0]
        # 1. 输入投影 (Embedding Layer)
        self.input_projs = nn.ModuleDict({
            nt: nn.Linear(in_side_dict[nt], hidden_size * 8) for nt in self.node_types
        })
        
        # 2. H2GCN 卷积层 (1阶邻居)
        self.h2_layer = H2GCN_Layer(metadata, hidden_size * 8, hidden_size * 8)
        
        # 3. 最终分类器 (处理拼接后的特征: 自身 + 1阶)
        # H2GCN 原文通过拼接多阶特征来捕捉异质性
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size * 8 * 2, 64), # 2倍维度因为 Concatenation
            nn.ReLU(),
            nn.Linear(64, out_size)
        )

    def forward(self, x_dict, edge_index_dict):
        # Eq. 4: 初始嵌入
        h_0 = {nt: F.relu(self.input_projs[nt](x)) for nt, x in x_dict.items()}
        
        # Eq. 5: 获取邻居聚合特征
        h_1 = self.h2_layer(h_0, edge_index_dict)
        
        # Eq. 6: 自身特征与邻居特征拼接 (H2GCN核心：分离同质与异质信息)
        # 我们关注 'review' 节点进行分类
        h_final = torch.cat([h_0['review'], h_1['review']], dim=-1)
        
        logits = self.classifier(h_final)
        return F.log_softmax(logits, dim=1)

# ==========================================
# 【模块三】：数据加载 (完全一致)
# ==========================================
def load_local_yelpchi():
    path = r'data/YelpChi/raw/YelpChi.mat'
    mat = sio.loadmat(path)
    data = HeteroData()
    data['review'].x = torch.tensor(mat['features'].todense()).float()
    data['review'].y = torch.tensor(mat['label'].flatten()).long()
    from torch_geometric.utils import from_scipy_sparse_matrix
    for key in ['net_rur', 'net_rsr', 'net_rtr']:
        edge_index, _ = from_scipy_sparse_matrix(mat[key])
        data['review', key.split('_')[1], 'review'].edge_index = edge_index
    return data

data = load_local_yelpchi().to(device)
in_size_dict = {nt: data[nt].x.size(1) for nt in data.node_types}
num_reviews = data['review'].num_nodes
indices = torch.randperm(num_reviews)
train_size, val_size = int(0.4 * num_reviews), int(0.2 * num_reviews)
train_idx, val_idx, test_idx = indices[:train_size], indices[train_size:train_size+val_size], indices[train_size+val_size:]

# H2GCN 模型初始化
model = H2GCN_Hetero(data.metadata(), in_size_dict, Config.hidden_dim, 2).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=Config.lr_init, weight_decay=Config.weight_decay)
scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=[30, 50], gamma=0.5)

labels = data['review'].y[train_idx]
weight_val = torch.tensor([1.0, ( (labels==0).sum()/(labels==1).sum() ) * 1.5]).to(device)

from torch_geometric.loader import NeighborLoader
train_loader = NeighborLoader(data, num_neighbors=[5, 4], batch_size=256,
                              input_nodes=('review', train_idx), shuffle=True, num_workers=4)

# ==========================================
# 【模块四】：评估逻辑 (统一动态阈值)
# ==========================================
scaler = GradScaler()

def evaluate_with_best_threshold(val_indices, test_indices):
    model.eval()
    with torch.no_grad():
        with autocast():
            logits = model(data.x_dict, data.edge_index_dict)
        probs = F.softmax(logits, dim=1)[:, 1].cpu().numpy()
        y_true = data['review'].y.cpu().numpy()
        val_y, val_probs = y_true[val_indices], probs[val_indices]
        
        best_threshold, max_val_f1 = 0.5, 0
        for threshold in np.linspace(0.05, 0.95, 91):
            current_f1 = f1_score(val_y, val_probs > threshold, average='macro')
            if current_f1 > max_val_f1:
                max_val_f1, best_threshold = current_f1, threshold
        
        test_y, test_probs = y_true[test_indices], probs[test_indices]
        test_pred = (test_probs > best_threshold).astype(int)
        test_auc = roc_auc_score(test_y, test_probs)
        test_f1 = f1_score(test_y, test_pred, average='macro')
        return test_auc, test_f1, best_threshold, max_val_f1, confusion_matrix(test_y, test_pred).ravel()

# ==========================================
# 【模块五】：训练循环
# ==========================================
early_stopper = EarlyStopping(patience=10)
best_valid_f1 = 0.0

for epoch in range(1, Config.epochs + 1):
    model.train()
    total_loss = 0
    for batch in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        with autocast():
            logits = model(batch.x_dict, batch.edge_index_dict)
            # 只计算中心 review 节点的损失
            loss = F.nll_loss(logits[:batch['review'].batch_size], 
                              batch['review'].y[:batch['review'].batch_size], weight=weight_val)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()
    
    scheduler.step()
    
    if epoch % 5 == 0:
        test_auc, test_f1, best_t, val_f1, cm = evaluate_with_best_threshold(val_idx, test_idx)
        if val_f1 > best_valid_f1:
            best_valid_f1 = val_f1
            status = "[*] NEW BEST"
        else: status = "[ ]"
        
        print(f"Ep: {epoch:03d} | Loss: {total_loss/len(train_loader):.4f} | Best_T: {best_t:.3f}")
        print(f"{status} Val_F1: {val_f1:.4f} | Test_AUC: {test_auc:.4f} | Test_F1: {test_f1:.4f}")
        
        early_stopper(val_f1)
        if early_stopper.early_stop: break

/home/malina/.venv/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/malina/.venv/lib/python3.8/site-packages/torch_geometric/sampler/neighbor_sampler.py:61: UserWarning: Using 'NeighborSampler' without a 'pyg-lib' installation is deprecated and will be removed soon. Please install 'pyg-lib' for accelerated neighborhood sampling
  warnings.warn(f"Using '{self.__class__.__name__}' without a "
/tmp/ipykernel_1333107/397745394.py:125: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
/tmp/ipykernel_1333107/397745394.py:160: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/tmp/ipykernel_1333107/397745394.py:130: 

Ep: 005 | Loss: 0.4491 | Best_T: 0.740
[*] NEW BEST Val_F1: 0.7260 | Test_AUC: 0.8680 | Test_F1: 0.7275


/tmp/ipykernel_1333107/397745394.py:160: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/tmp/ipykernel_1333107/397745394.py:130: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Ep: 010 | Loss: 0.3861 | Best_T: 0.790
[*] NEW BEST Val_F1: 0.7676 | Test_AUC: 0.9058 | Test_F1: 0.7716


/tmp/ipykernel_1333107/397745394.py:160: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/tmp/ipykernel_1333107/397745394.py:130: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


...

Ep: 040 | Loss: 0.2679 | Best_T: 0.830
[ ] Val_F1: 0.7788 | Test_AUC: 0.9144 | Test_F1: 0.7888


/tmp/ipykernel_1333107/397745394.py:160: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/tmp/ipykernel_1333107/397745394.py:130: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Ep: 045 | Loss: 0.2491 | Best_T: 0.820
[*] NEW BEST Val_F1: 0.7851 | Test_AUC: 0.9158 | Test_F1: 0.7939


/tmp/ipykernel_1333107/397745394.py:160: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


KeyboardInterrupt: 

In [ ]:
#单层 H2GCN_Hetero

# YelpChi 数据集 Baseline: H2GCN (Single Layer Version)
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import scipy.io as sio 
from torch_geometric.data import HeteroData  
from torch_geometric.nn import HeteroConv, GCNConv
from torch.cuda.amp import autocast, GradScaler 
from sklearn.metrics import confusion_matrix, f1_score, roc_auc_score

# ==========================================
# 【模块一】：Config 参数 (保持对齐)
# ==========================================
class Config:
    hidden_dim = 32      
    epochs = 100         
    lr_init = 0.001
    weight_decay = 1e-4

class EarlyStopping:
    def __init__(self, patience=15, min_delta=0):
        self.patience, self.min_delta = patience, min_delta
        self.counter, self.best_score, self.early_stop = 0, None, False
    def __call__(self, current_f1):
        if self.best_score is None: self.best_score = current_f1
        elif current_f1 < self.best_score + self.min_delta:
            self.counter += 1
            if self.counter >= self.patience: self.early_stop = True
        else:
            self.best_score, self.counter = current_f1, 0

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ==========================================
# 【模块二】：H2GCN 核心架构 (单层实现)
# ==========================================
class H2GCN_Layer(nn.Module):
    """ 实现 H2GCN 的异质扩展：仅聚合一阶邻居特征 """
    def __init__(self, metadata, in_channels, out_channels):
        super().__init__()
        self.conv = HeteroConv({
            rel: GCNConv(in_channels, out_channels, add_self_loops=False)
            for rel in metadata[1]
        }, aggr='sum')

    def forward(self, x_dict, edge_index_dict):
        h_neighbors = self.conv(x_dict, edge_index_dict)
        return h_neighbors

class H2GCN_Hetero(nn.Module):
    def __init__(self, metadata, in_side_dict, hidden_size, out_size):
        super().__init__()
        self.node_types = metadata[0]
        # 1. 输入投影 (H0)
        self.input_projs = nn.ModuleDict({
            nt: nn.Linear(in_side_dict[nt], hidden_size * 8) for nt in self.node_types
        })
        
        # 2. 单层 H2GCN 卷积 (仅获取 1-hop 邻居)
        self.h2_layer = H2GCN_Layer(metadata, hidden_size * 8, hidden_size * 8)
        
        # 3. 最终分类器: 输入为 [h_0 || h_1]
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size * 8 * 2, 64), 
            nn.ReLU(),
            nn.Linear(64, out_size)
        )

    def forward(self, x_dict, edge_index_dict):
        # Step 1: 自身特征投影 (ego-feature)
        h_0 = {nt: F.relu(self.input_projs[nt](x)) for nt, x in x_dict.items()}
        
        # Step 2: 邻居聚合 (1-hop neighbor-feature)
        h_1 = self.h2_layer(h_0, edge_index_dict)
        
        # Step 3: 特征拼接 (H2GCN核心：分离自身与环境信息)
        # 针对 review 节点进行分类，特征为 [自身特征 || 邻居特征]
        # 如果某些 nt 在当前 batch 没有被聚合到，h_1 可能会缺 key，这里用 h_filtered 逻辑兼容
        h_1_review = h_1.get('review', torch.zeros_like(h_0['review']))
        h_final = torch.cat([h_0['review'], h_1_review], dim=-1)
        
        logits = self.classifier(h_final)
        return F.log_softmax(logits, dim=1)

# ==========================================
# 【模块三】：数据加载 (保持一致)
# ==========================================
def load_local_yelpchi():
    path = r'data/YelpChi/raw/YelpChi.mat'
    mat = sio.loadmat(path)
    data = HeteroData()
    data['review'].x = torch.tensor(mat['features'].todense()).float()
    data['review'].y = torch.tensor(mat['label'].flatten()).long()
    from torch_geometric.utils import from_scipy_sparse_matrix
    for key in ['net_rur', 'net_rsr', 'net_rtr']:
        edge_index, _ = from_scipy_sparse_matrix(mat[key])
        data['review', key.split('_')[1], 'review'].edge_index = edge_index
    return data

data = load_local_yelpchi().to(device)
in_size_dict = {nt: data[nt].x.size(1) for nt in data.node_types}
num_reviews = data['review'].num_nodes
indices = torch.randperm(num_reviews)
train_size, val_size = int(0.4 * num_reviews), int(0.2 * num_reviews)
train_idx, val_idx, test_idx = indices[:train_size], indices[train_size:train_size+val_size], indices[train_size+val_size:]

# 模型初始化
model = H2GCN_Hetero(data.metadata(), in_size_dict, Config.hidden_dim, 2).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=Config.lr_init, weight_decay=Config.weight_decay)
scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=[30, 50], gamma=0.5)

labels = data['review'].y[train_idx]
weight_val = torch.tensor([1.0, ( (labels==0).sum()/(labels==1).sum() ) * 1.5]).to(device)

from torch_geometric.loader import NeighborLoader
train_loader = NeighborLoader(data, num_neighbors=[15], batch_size=256,
                              input_nodes=('review', train_idx), shuffle=True, num_workers=4)

# ==========================================
# 【模块四】：评估逻辑 (统一动态阈值)
# ==========================================
scaler = GradScaler()

def evaluate_with_best_threshold(val_indices, test_indices):
    model.eval()
    with torch.no_grad():
        with autocast():
            logits = model(data.x_dict, data.edge_index_dict)
        probs = F.softmax(logits, dim=1)[:, 1].cpu().numpy()
        y_true = data['review'].y.cpu().numpy()
        val_y, val_probs = y_true[val_indices], probs[val_indices]
        
        best_threshold, max_val_f1 = 0.5, 0
        for threshold in np.linspace(0.05, 0.95, 91):
            current_f1 = f1_score(val_y, val_probs > threshold, average='macro')
            if current_f1 > max_val_f1:
                max_val_f1, best_threshold = current_f1, threshold
        
        test_y, test_probs = y_true[test_indices], probs[test_indices]
        test_pred = (test_probs > best_threshold).astype(int)
        test_auc = roc_auc_score(test_y, test_probs)
        test_f1 = f1_score(test_y, test_pred, average='macro')
        return test_auc, test_f1, best_threshold, max_val_f1, confusion_matrix(test_y, test_pred).ravel()

# ==========================================
# 【模块五】：训练循环
# ==========================================
early_stopper = EarlyStopping(patience=10)
best_valid_f1 = 0.0

for epoch in range(1, Config.epochs + 1):
    model.train()
    total_loss = 0
    for batch in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        with autocast():
            logits = model(batch.x_dict, batch.edge_index_dict)
            loss = F.nll_loss(logits[:batch['review'].batch_size], 
                              batch['review'].y[:batch['review'].batch_size], weight=weight_val)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()
    
    scheduler.step()
    
    if epoch % 5 == 0:
        test_auc, test_f1, best_t, val_f1, cm = evaluate_with_best_threshold(val_idx, test_idx)
        if val_f1 > best_valid_f1:
            best_valid_f1 = val_f1
            status = "[*] NEW BEST"
        else: status = "[ ]"
        
        print(f"Ep: {epoch:03d} | Loss: {total_loss/len(train_loader):.4f} | Best_T: {best_t:.3f}")
        print(f"{status} Val_F1: {val_f1:.4f} | Test_AUC: {test_auc:.4f} | Test_F1: {test_f1:.4f}")
        
        early_stopper(val_f1)
        if early_stopper.early_stop: break

/tmp/ipykernel_1393756/1359116282.py:147: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
/tmp/ipykernel_1393756/1359116282.py:182: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/tmp/ipykernel_1393756/1359116282.py:152: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


ValueError: Input contains NaN.

In [ ]:
# YelpChi 数据集 Baseline: HAN (Heterogeneous Graph Attention Network)

import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import scipy.io as sio 
from torch_geometric.data import HeteroData  
from torch_geometric.nn import HANConv # 引入 HAN 算子
from torch.cuda.amp import autocast, GradScaler 
from sklearn.metrics import confusion_matrix, f1_score, roc_auc_score

# ==========================================
# 【模块一】：Config 参数 (保持不变)
# ==========================================
class Config:
    hidden_dim = 32      
    epochs = 100         
    lr_init = 0.001
    weight_decay = 1e-4

# EarlyStopping 类保持不变...
class EarlyStopping:
    def __init__(self, patience=15, min_delta=0):
        self.patience, self.min_delta = patience, min_delta
        self.counter, self.best_score, self.early_stop = 0, None, False
    def __call__(self, current_f1):
        if self.best_score is None: self.best_score = current_f1
        elif current_f1 < self.best_score + self.min_delta:
            self.counter += 1
            if self.counter >= self.patience: self.early_stop = True
        else:
            self.best_score, self.counter = current_f1, 0

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ==========================================
# 【模块二】：HAN 核心架构
# ==========================================
class HAN_Hetero(nn.Module):
    def __init__(self, metadata, in_side_dict, hidden_size, out_size):
        super().__init__()
        self.node_types, self.edge_types = metadata
        self.total_hidden = hidden_size * 8
        
        # 输入投影
        self.input_projs = nn.ModuleDict({
            nt: nn.Linear(in_side_dict[nt], self.total_hidden) for nt in self.node_types
        })
        
        # [修改点]：定义两层 HANConv
        self.han1 = HANConv(self.total_hidden, self.total_hidden, metadata, heads=8, dropout=0.1)
        self.han2 = HANConv(self.total_hidden, self.total_hidden, metadata, heads=8, dropout=0.1)
        
        self.classifier = nn.Sequential(
            nn.Linear(self.total_hidden, 64),
            nn.ReLU(),
            nn.Linear(64, out_size)
        )

    def forward(self, x_dict, edge_index_dict):
        # 1. 投影
        h_dict = {nt: F.elu(self.input_projs[nt](x)) for nt, x in x_dict.items()}
        
        # 2. 第一层 HAN
        h_dict = self.han1(h_dict, edge_index_dict)
        h_dict = {nt: F.elu(h) for nt, h in h_dict.items()} # 加入激活函数
        
        # 3. 第二层 HAN
        h_dict = self.han2(h_dict, edge_index_dict)
        
        logits = self.classifier(h_dict['review'])
        return F.log_softmax(logits, dim=1)

# ==========================================
# 【模块三】：数据加载 (完全一致)
# ==========================================
def load_local_yelpchi():
    path = r'data/YelpChi/raw/YelpChi.mat'
    mat = sio.loadmat(path)
    data = HeteroData()
    data['review'].x = torch.tensor(mat['features'].todense()).float()
    data['review'].y = torch.tensor(mat['label'].flatten()).long()
    from torch_geometric.utils import from_scipy_sparse_matrix
    for key in ['net_rur', 'net_rsr', 'net_rtr']:
        edge_index, _ = from_scipy_sparse_matrix(mat[key])
        data['review', key.split('_')[1], 'review'].edge_index = edge_index
    return data

data = load_local_yelpchi().to(device)
in_size_dict = {nt: data[nt].x.size(1) for nt in data.node_types}
num_reviews = data['review'].num_nodes
indices = torch.randperm(num_reviews)
train_size, val_size = int(0.4 * num_reviews), int(0.2 * num_reviews)
train_idx, val_idx, test_idx = indices[:train_size], indices[train_size:train_size+val_size], indices[train_size+val_size:]

# HAN 模型初始化
model = HAN_Hetero(data.metadata(), in_size_dict, Config.hidden_dim, 2).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=Config.lr_init, weight_decay=Config.weight_decay)
scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=[30, 50], gamma=0.5)

labels = data['review'].y[train_idx]
weight_val = torch.tensor([1.0, ( (labels==0).sum()/(labels==1).sum() ) * 1.5]).to(device)

from torch_geometric.loader import NeighborLoader
train_loader = NeighborLoader(data, num_neighbors=[5, 4], batch_size=256,
                              input_nodes=('review', train_idx), shuffle=True, num_workers=4)

# ==========================================
# 【模块四】：评估逻辑 (保持对齐)
# ==========================================
scaler = GradScaler()

def evaluate_with_best_threshold(val_indices, test_indices):
    model.eval()
    with torch.no_grad():
        with autocast():
            logits = model(data.x_dict, data.edge_index_dict)
        probs = F.softmax(logits, dim=1)[:, 1].cpu().numpy()
        y_true = data['review'].y.cpu().numpy()
        val_y, val_probs = y_true[val_indices], probs[val_indices]
        
        best_threshold, max_val_f1 = 0.5, 0
        for threshold in np.linspace(0.05, 0.95, 91):
            current_f1 = f1_score(val_y, val_probs > threshold, average='macro')
            if current_f1 > max_val_f1:
                max_val_f1, best_threshold = current_f1, threshold
        
        test_y, test_probs = y_true[test_indices], probs[test_indices]
        test_pred = (test_probs > best_threshold).astype(int)
        test_auc = roc_auc_score(test_y, test_probs)
        test_f1 = f1_score(test_y, test_pred, average='macro')
        return test_auc, test_f1, best_threshold, max_val_f1, confusion_matrix(test_y, test_pred).ravel()

# ==========================================
# 【模块五】：训练循环
# ==========================================
early_stopper = EarlyStopping(patience=10)
best_valid_f1 = 0.0

for epoch in range(1, Config.epochs + 1):
    model.train()
    total_loss = 0
    for batch in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        with autocast():
            logits = model(batch.x_dict, batch.edge_index_dict)
            loss = F.nll_loss(logits[:batch['review'].batch_size], 
                              batch['review'].y[:batch['review'].batch_size], weight=weight_val)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()
    
    scheduler.step()
    
    if epoch % 5 == 0:
        test_auc, test_f1, best_t, val_f1, cm = evaluate_with_best_threshold(val_idx, test_idx)
        if val_f1 > best_valid_f1:
            best_valid_f1 = val_f1
            status = "[*] NEW BEST"
        else: status = "[ ]"
        
        print(f"Ep: {epoch:03d} | Loss: {total_loss/len(train_loader):.4f} | Best_T: {best_t:.3f}")
        print(f"{status} Val_F1: {val_f1:.4f} | Test_AUC: {test_auc:.4f} | Test_F1: {test_f1:.4f}")
        
        early_stopper(val_f1)
        if early_stopper.early_stop: break

/home/malina/.venv/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/malina/.venv/lib/python3.8/site-packages/torch_geometric/sampler/neighbor_sampler.py:61: UserWarning: Using 'NeighborSampler' without a 'pyg-lib' installation is deprecated and will be removed soon. Please install 'pyg-lib' for accelerated neighborhood sampling
  warnings.warn(f"Using '{self.__class__.__name__}' without a "
/tmp/ipykernel_1500513/714361581.py:113: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
/tmp/ipykernel_1500513/714361581.py:148: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/tmp/ipykernel_1500513/714361581.py:118: 

Ep: 005 | Loss: 0.5304 | Best_T: 0.700
[*] NEW BEST Val_F1: 0.5483 | Test_AUC: 0.7418 | Test_F1: 0.5437


/tmp/ipykernel_1500513/714361581.py:148: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/tmp/ipykernel_1500513/714361581.py:118: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Ep: 010 | Loss: 0.5091 | Best_T: 0.700
[*] NEW BEST Val_F1: 0.5515 | Test_AUC: 0.7658 | Test_F1: 0.5491


/tmp/ipykernel_1500513/714361581.py:148: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/tmp/ipykernel_1500513/714361581.py:118: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


...

Ep: 025 | Loss: 0.4819 | Best_T: 0.760
[ ] Val_F1: 0.5488 | Test_AUC: 0.7517 | Test_F1: 0.5451


/tmp/ipykernel_1500513/714361581.py:148: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/tmp/ipykernel_1500513/714361581.py:118: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Ep: 030 | Loss: 0.4627 | Best_T: 0.760
[ ] Val_F1: 0.5418 | Test_AUC: 0.7437 | Test_F1: 0.5365


/tmp/ipykernel_1500513/714361581.py:148: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


KeyboardInterrupt: 

In [ ]:
# YelpChi 数据集 Baseline: Hetero-GAT + Random DropEdge (Sparsity=0.3 对齐)
# RDG

import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import scipy.io as sio
import random
from torch_geometric.data import HeteroData
from torch_geometric.nn import GATConv, HeteroConv
from torch_geometric.utils import dropout_edge
from sklearn.metrics import confusion_matrix, roc_auc_score, f1_score

# ==========================================
# 【模块一】：Config 参数 (对齐主实验)
# ==========================================
class Config:
    hidden_dim = 32      # 单头维度
    num_heads = 8       # 拼接后 32 * 8 = 256 维，对齐 NRS 的 total_hidden
    epochs = 150
    lr_init = 0.001
    weight_decay = 1e-4
    target_sparsity = 0.30  # 随机保留 30% 的边
    seeds = [1, 7, 314, 1227, 2026]

class EarlyStopping:
    def __init__(self, patience=15):
        self.patience = patience
        self.counter = 0
        self.best_score = None
        self.early_stop = False

    def __call__(self, current_f1):
        if self.best_score is None:
            self.best_score = current_f1
        elif current_f1 <= self.best_score:
            self.counter += 1
            if self.counter >= self.patience: self.early_stop = True
        else:
            self.best_score = current_f1
            self.counter = 0

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ==========================================
# 【模块二】：DropEdge 适配型模型
# ==========================================
class HeteroGAT_DropEdge(nn.Module):
    def __init__(self, metadata, in_side_dict, head_dim, num_heads, out_size):
        super().__init__()
        self.node_types, self.edge_types = metadata
        
        # 1. 初始特征投影 (对齐 NRS 的 input_projs)
        self.input_projs = nn.ModuleDict({
            nt: nn.Linear(in_side_dict[nt], head_dim * num_heads) 
            for nt in self.node_types
        })
        
        # 2. 两层异质 GAT (对齐 NRS 的 convs)
        self.convs = nn.ModuleList([
            HeteroConv({
                rel: GATConv(head_dim * num_heads, head_dim, heads=num_heads, add_self_loops=False)
                for rel in self.edge_types
            }, aggr='sum') for _ in range(2)
        ])
        
        # 3. 分类器 (对齐 NRS 的 classifier)
        self.classifier = nn.Sequential(
            nn.Linear(head_dim * num_heads, 64),
            nn.ReLU(),
            nn.Linear(64, out_size)
        )

    def forward(self, x_dict, edge_index_dict):
        # 投影到对齐维度
        h_dict = {nt: F.elu(self.input_projs[nt](x)) for nt, x in x_dict.items()}
        
        # 两层消息传递
        for i in range(2):
            h_dict = self.convs[i](h_dict, edge_index_dict)
            h_dict = {nt: F.elu(h) for nt, h in h_dict.items()}
            
        logits = self.classifier(h_dict['review'])
        return F.log_softmax(logits, dim=1)

# ==========================================
# 【模块三】：工具函数
# ==========================================
def apply_dropedge(edge_index_dict, p):
    """ 对异质图中的每一类关系应用随机边删除 """
    new_edge_index_dict = {}
    for rel, edge_index in edge_index_dict.items():
        new_index, _ = dropout_edge(edge_index, p=p)
        new_edge_index_dict[rel] = new_index
    return new_edge_index_dict

def load_local_yelpchi():
    path = r'data/YelpChi/raw/YelpChi.mat'
    mat = sio.loadmat(path)
    data = HeteroData()
    data['review'].x = torch.tensor(mat['features'].todense()).float()
    data['review'].y = torch.tensor(mat['label'].flatten()).long()
    from torch_geometric.utils import from_scipy_sparse_matrix
    for key in ['net_rur', 'net_rsr', 'net_rtr']:
        edge_index, _ = from_scipy_sparse_matrix(mat[key])
        data['review', key.split('_')[1], 'review'].edge_index = edge_index
    return data

# ==========================================
# 【模块四】：实验运行逻辑
# ==========================================
def run_experiment(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    
    data = load_local_yelpchi().to(device)
    num_nodes = data['review'].num_nodes
    indices = torch.randperm(num_nodes)
    train_size, val_size = int(0.4 * num_nodes), int(0.2 * num_nodes)
    train_idx, val_idx, test_idx = indices[:train_size], indices[train_size:train_size+val_size], indices[train_size+val_size:]

    model = HeteroGAT_DropEdge(data.metadata(), 
                               {nt: data[nt].x.size(1) for nt in data.node_types},
                               Config.hidden_dim, Config.num_heads, 2).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=Config.lr_init, weight_decay=Config.weight_decay)
    early_stopper = EarlyStopping(patience=15)
    
    labels = data['review'].y[train_idx]
    weight_val = torch.tensor([1.0, ((labels==0).sum()/(labels==1).sum()) * 1.5]).to(device)
    
    best_val_f1 = 0
    final_test_results = None

    for epoch in range(1, Config.epochs + 1):
        model.train()
        optimizer.zero_grad()
        
        # 训练阶段应用随机 DropEdge (保留 30%)
        train_edges = apply_dropedge(data.edge_index_dict, p=1-Config.target_sparsity)
        logits = model(data.x_dict, train_edges)
        
        loss = F.nll_loss(logits[train_idx], data['review'].y[train_idx], weight=weight_val)
        loss.backward()
        optimizer.step()

        if epoch % 5 == 0:
            model.eval()
            with torch.no_grad():
                # 推理阶段同样保持 30% 稀疏度以满足公平性对齐
                eval_edges = apply_dropedge(data.edge_index_dict, p=1-Config.target_sparsity)
                logits = model(data.x_dict, eval_edges)
                probs = F.softmax(logits, dim=1)[:, 1].cpu().numpy()
                y_true = data['review'].y.cpu().numpy()

                # 动态阈值搜索 (对齐主实验)
                val_probs, val_y = probs[val_idx], y_true[val_idx]
                best_t, max_f1 = 0.5, 0
                for t in np.linspace(0.05, 0.95, 91):
                    f = f1_score(val_y, val_probs > t, average='macro')
                    if f > max_f1: max_f1, best_t = f, t
                
                # 测试集汇报
                test_probs, test_y = probs[test_idx], y_true[test_idx]
                t_f1 = f1_score(test_y, test_probs > best_t, average='macro')
                t_auc = roc_auc_score(test_y, test_probs)
                
                if max_f1 > best_val_f1:
                    best_val_f1 = max_f1
                    final_test_results = (t_auc, t_f1, best_t)
                    status = "[*] NEW BEST"
                else: status = "[ ]"

                print(f"Seed: {seed} | Ep: {epoch:03d} | Val_F1: {max_f1:.4f} | Test_F1: {t_f1:.4f} {status}")
            
            early_stopper(max_f1)
            if early_stopper.early_stop: break
            
    return final_test_results

if __name__ == "__main__":
    all_metrics = []
    print(f"Starting YelpChi Baseline: GAT + DropEdge (Sparsity: {Config.target_sparsity})")
    for s in Config.seeds:
        res = run_experiment(s)
        all_metrics.append(res)
    
    aucs = [m[0] for m in all_metrics]
    f1s = [m[1] for m in all_metrics]
    print(f"\nFinal Results (Average over 5 seeds):")
    print(f"AUC: {np.mean(aucs)*100:.2f} ± {np.std(aucs)*100:.2f}")
    print(f"F1 : {np.mean(f1s)*100:.2f} ± {np.std(f1s)*100:.2f}")